In [ ]:
import numpy as np
import torch
import torch.nn as nn

In [ ]:
class PINN(nn.Module):
    def __init__(self, N_INPUT, N_OUTPUT, N_HIDDEN, N_LAYERS): #for 2D heat equation , N_INPUT is 3 and N_OUTPUT is 1(temperture)
      super().__init__()
      activation=nn.Tanh
      self.fcs = nn.Sequential(*[nn.Linear(N_INPUT, N_HIDDEN),activation()])
      self.fch = nn.Sequential(*[nn.Sequential(*[nn.Linear(N_HIDDEN, N_HIDDEN),activation()]) for _ in range(N_LAYERS-1)])
      self.fce = nn.Linear(N_HIDDEN, N_OUTPUT)

In [ ]:
torch.manual_seed(123)
pinn=PINN(3,1,100,3)
a,b=4,4 # width and length of the 2d Shape
T_i=37 #initial temperture at t=0
normalized_Ti=1
a=0.000002 #thermal diffusitivity of iron ,a=λ/(ρ*C_p)
##collocationpoints
# Number of points
N_f_int = 4000   # interior collocation points
N_f_bnd = 500    # boundary points per face
N_f_init = 500   # initial condition points

x_min,x_max=0,1
y_min,y_max=0,1
t_min,t_max=0,1
# --- Interior points (random uniform) ---
x_int=torch.rand(N_f_int,1)*(x_max-x_min)+x_min
y_int=torch.rand(N_f_int,1)*(y_max-y_min)+y_min
t_int=torch.rand(N_f_int,1)*(t_max-t_min)+t_min
##initial conditions:
x_init=np.linspace(x_min,x_max,N_f_init)
y_init=np.linspace(y_min,y_max,N_f_init)
t_init=np.zeros(N_f_init)
##boundary condition x=0
y_bnd=np.linspace(y_min,y_max,N_f_bnd)
t_bnd=np.linspace(t_min,t_max,N_f_bnd)
x_bnd=np.zeros_like(y_bnd)
##boundary condition y=0
x_bnd=np.linspace(x_min,x_max,N_f_bnd)
t_bnd=np.linspace(t_min,t_max,N_f_bnd)
y_bnd=np.zeros_like(x_bnd)
## convert to [x,y,t] tensors
initial_points=torch.tensor(np.stack([x_init,y_init,t_init],axis=1),dtype=torch.float32)
boundary_points=torch.tensor(np.stack([x_bnd,y_bnd,t_bnd],axis=1),dtype=torch.float32)
